In [ ]:
!pip install gradio groq -q

In [ ]:
import gradio as gr
from groq import Groq
import json
from google.colab import userdata

SYSTEM_PROMPT = """You are ContextBridge, an expert communication analyst. Analyze workplace communication and return ONLY a JSON object — no extra text, no markdown, just raw JSON.

Return exactly this structure:
{
  "risk_score": <integer 0-100>,
  "risk_label": "<Low|Medium|High|Critical>",
  "what_writer_meant": "<1-2 sentences explaining the real intent>",
  "what_reader_will_think": [
    "<misinterpretation 1>",
    "<misinterpretation 2>",
    "<misinterpretation 3>"
  ],
  "problem_phrases": [
    "<exact phrase from text causing confusion>",
    "<exact phrase from text causing confusion>"
  ],
  "rewrite": "<full rewritten version that preserves the writer's tone>",
  "explanation": "<1 sentence on why this rewrite is clearer>"
}

Risk score: 0-30=Low, 31-60=Medium, 61-85=High, 86-100=Critical"""


def analyze(text, api_key):
    if not text.strip():
        return "Please enter some text.", "", "", "", ""

    key = api_key.strip() if api_key.strip() else userdata.get("GROQ_API_KEY")
    if not key:
        return "Please enter your Groq API key.", "", "", "", ""

    try:
        client = Groq(api_key=key)
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Analyze this workplace communication:\n\n{text}"}
            ],
            temperature=0.3,
            max_tokens=1000
        )

        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()

        data = json.loads(raw)
        score = data["risk_score"]
        label = data["risk_label"]

        color = "🟢" if score <= 30 else "🟡" if score <= 60 else "🔴" if score <= 85 else "🚨"
        score_out = f"{color} {score}/100 — {label} Miscommunication Risk"
        intent_out = f"What the writer actually meant:\n{data['what_writer_meant']}"
        misread_out = "How readers will misinterpret this:\n" + "".join(f"\n{i}. {m}" for i, m in enumerate(data["what_reader_will_think"], 1))
        phrases_out = "Problem phrases:\n" + "".join(f'\n→ "{p}"' for p in data["problem_phrases"])
        rewrite_out = f"{data['rewrite']}\n\n— Why this works: {data['explanation']}"

        return score_out, intent_out, misread_out, phrases_out, rewrite_out

    except json.JSONDecodeError:
        return "Could not parse response. Try again.", "", "", "", ""
    except Exception as e:
        return f"Error: {str(e)}", "", "", "", ""


EXAMPLES = [
    ["Fix the thing from last week asap its blocking everything"],
    ["Can you look at the doc I sent and let me know"],
    ["The client is not happy. Please handle it."],
    ["We need to discuss the project. Its urgent."],
]

with gr.Blocks(title="ContextBridge", theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div style="text-align:center; padding:20px 0 8px;">
        <h1 style="font-size:26px; font-weight:700;">ContextBridge</h1>
        <p style="color:#6b7280; font-size:14px; max-width:500px; margin:0 auto;">
        World's first AI that detects the gap between what you <em>meant</em> and what you <em>wrote</em>
        </p>
    </div>
    """)
    api_input = gr.Textbox(label="Groq API Key (free at console.groq.com)", placeholder="gsk_...", type="password", lines=1)
    text_input = gr.Textbox(label="Paste any workplace message", placeholder="e.g. Fix the thing from last week asap its blocking prod", lines=5)
    analyze_btn = gr.Button("Detect Communication Gap", variant="primary", size="lg")
    gr.Examples(examples=EXAMPLES, inputs=text_input, label="Try these examples")
    score_out = gr.Textbox(label="Risk Score", lines=1)
    intent_out = gr.Textbox(label="What writer actually meant", lines=3)
    misread_out = gr.Textbox(label="How reader will misinterpret this", lines=5)
    phrases_out = gr.Textbox(label="Problem phrases", lines=3)
    rewrite_out = gr.Textbox(label="ContextBridge rewrite", lines=5)

    analyze_btn.click(fn=analyze, inputs=[text_input, api_input], outputs=[score_out, intent_out, misread_out, phrases_out, rewrite_out])

demo.launch(share=True, debug=True)

/tmp/ipykernel_28762/1485672151.py:82: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="ContextBridge", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://03d89bf80c7a31f7cb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
